# Self-Supervised Model Inspection

本笔记本用于自监督模型的推理与可视化：展示网络输出（重建/obj）、预测的 Zernike 系数、RMS、Zernike 相位图，以及对应的真实 obj 图像、真实 Zernike 相位图与系数。

参照：inspect_supervised_model.ipynb 的结构，必要时修改路径和数据字段以配合你的数据集/模型输出。

## 1. 环境与依赖
如果缺少依赖，请在环境中安装，例如：
```bash
pip install -e . matplotlib torch numpy
```

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

# 尝试导入项目工具（根据工程实际模块名调整）
try:
    from ao2d.models.factory import make_model
    from ao2d.data import AO2DSelfDataset, build_dataloader, get_data_root, resolve_path
    from ao2d.optics import zernike_wavefront, pupil_coordinates_2d, generate_psf2d_from_zernike
    HAS_PROJECT = True
except Exception:
    HAS_PROJECT = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('repo:', REPO_ROOT)
print('device:', device)

repo: /data/yxdeng993/code/SSPAO2D
device: cuda


In [ ]:
# Zernike helper: use SSPA O2D optics implementations
import numpy as np
import torch
from ao2d.optics import zernike_wavefront, pupil_coordinates_2d


def zernike_mode_count(zernike_indices=None, optics_cfg=None):
    if zernike_indices is not None:
        return len(tuple(zernike_indices))
    if optics_cfg and optics_cfg.get('zernike_indices') is not None:
        return len(optics_cfg['zernike_indices'])
    return 13


def as_zernike_coeffs(coeffs, zernike_indices=None, optics_cfg=None):
    """Return coefficients; None means no aberration (all zeros)."""
    if coeffs is None:
        return np.zeros(zernike_mode_count(zernike_indices, optics_cfg), dtype=np.float32)
    return np.asarray(coeffs).squeeze()


def coeffs_to_wavefront(
    coeffs,
    image_size,
    zernike_indices=None,
    optics_cfg=None,
    device='cpu',
    dtype=torch.float32,
):
    """Compute Zernike OPD wavefront (micrometers) from coefficients."""
    if zernike_indices is None:
        zernike_indices = tuple(range(3, 16))
    optics_cfg = optics_cfg or {}
    pixel_size = float(optics_cfg.get('pixel_size', 0.065))
    wavelength = float(optics_cfg.get('lambda_emission', 0.525))
    na = float(optics_cfg.get('na', 1.0))
    coeffs = as_zernike_coeffs(coeffs, zernike_indices, optics_cfg)
    coeffs_t = torch.as_tensor(coeffs, device=device, dtype=dtype)
    if coeffs_t.ndim == 1:
        coeffs_t = coeffs_t.unsqueeze(0)
    rho, theta, pupil = pupil_coordinates_2d(
        image_size,
        pixel_size=pixel_size,
        wavelength=wavelength,
        na=na,
        device=coeffs_t.device,
        dtype=dtype,
    )
    wf = zernike_wavefront(zernike_indices, coeffs_t, rho, theta)
    return wf.squeeze(0).cpu().numpy(), pupil.cpu().numpy()


def coeffs_rms(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    return np.sqrt(np.mean((a - b) ** 2))

In [ ]:
# 配置：根据需要修改路径/文件名
CONFIG_PATH = REPO_ROOT / 'configs' / 'self_supervised_2d.json'
CHECKPOINT_NAME = 'best.pt'
OUTPUT_DIR = None
DATA_ROOT_OVERRIDE = None  # 可改为你的数据根路径
MAX_EVAL_SAMPLES = 8
N_ZERNIKE = None  # 将在加载 cfg 后自动读取
OPTICS_CFG = None  # 将在加载 cfg 后自动读取

In [ ]:
# 加载模型与数据（按项目实际 API 调整）
if not HAS_PROJECT:
    print('未检测到项目模块，跳过自动加载。请手工构建 model 和 eval_loader。')
    model = None
    eval_loader = None
else:
    import json
    with open(CONFIG_PATH, 'r') as f:
        cfg = json.load(f)
    OPTICS_CFG = cfg.get('optics', {})
    N_ZERNIKE = int(
        cfg.get('model', {}).get(
            'zernike_modes',
            len(cfg.get('model', {}).get('zernike_indices', OPTICS_CFG.get('zernike_indices', list(range(3, 16))))),
        )
    )
    output_dir = Path(cfg.get('output_dir', 'outputs/selfsupervised')) if OUTPUT_DIR is None else Path(OUTPUT_DIR)
    ckpt = output_dir / CHECKPOINT_NAME
    print('checkpoint:', ckpt, ckpt.exists())
    print('zernike_modes:', N_ZERNIKE)
    print('optics cfg:', OPTICS_CFG)
    model = make_model(cfg['model'])
    state = torch.load(ckpt, map_location='cpu') if ckpt.exists() else {}
    if 'model' in state:
        model.load_state_dict(state['model'], strict=False)
    model = model.to(device).eval()

    # 构造评估集：自监督配置使用 image_dir，不要求目标图像或真实 Zernike 系数
    try:
        split = 'test' if 'test' in cfg.get('data', {}) else 'val'
        data_root = get_data_root(cfg, DATA_ROOT_OVERRIDE)
        eval_set = AO2DSelfDataset(resolve_path(cfg['data'][split]['image_dir'], data_root), patch_size=tuple(cfg.get('data', {}).get('patch_size', [256,256])), augment=False, samples_per_epoch=cfg['data'][split].get('samples_per_epoch'))
        eval_loader = build_dataloader(eval_set, batch_size=1, shuffle=False, num_workers=0, drop_last=False)
        print('eval samples:', len(eval_set))
    except Exception as e:
        print('无法自动构建 eval_loader:', e)
        eval_loader = None

## 2. 推理：获取网络输出与预测 Zernike 系数（示例流程）

In [ ]:
def first_present(mapping, *keys):
    for key in keys:
        if key in mapping and mapping[key] is not None:
            return mapping[key]
    return None


def to_cpu_or_none(value):
    if value is None:
        return None
    return value.detach().cpu() if torch.is_tensor(value) else np.asarray(value)


examples = []
if eval_loader is None or model is None:
    print('请先确保 model 与 eval_loader 可用，然后重跑此单元。')
else:
    for idx, batch in enumerate(eval_loader):
        if idx >= MAX_EVAL_SAMPLES:
            break
        x = first_present(batch, 'input', 'aberrated', 'image')
        y = first_present(batch, 'target', 'obj', 'gt')
        x = x.to(device) if torch.is_tensor(x) else torch.tensor(x).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(x)
        # 解析模型输出：常见模式 -> (recon, coeffs) 或 单个 tensor 为 recon，coeffs 通过 model.attribute 获取
        if isinstance(out, (list, tuple)) and len(out) == 2:
            recon, pred_coeffs = out
        else:
            recon = out if torch.is_tensor(out) else out[0]
            # 尝试从模型上读取预测系数（约定）
            pred_coeffs = getattr(model, 'pred_coeffs', None)
            if pred_coeffs is None:
                # 有些模型在 forward 返回字典
                if isinstance(out, dict) and 'coeffs' in out:
                    pred_coeffs = out['coeffs']
                else:
                    pred_coeffs = None
        # 提取真实系数（如果数据集提供）
        true_coeffs = first_present(batch, 'zernike', 'coeffs', 'target_coeffs')
        examples.append({'x': x.cpu(), 'recon': recon.cpu(), 'y': to_cpu_or_none(y), 'pred_coeffs': to_cpu_or_none(pred_coeffs), 'true_coeffs': to_cpu_or_none(true_coeffs), 'meta': {'idx': idx, 'input_path': batch.get('input_path'), 'target_path': batch.get('target_path')}})
    print(f'evaluated {len(examples)} samples')

## 3. 可视化：网络输出、Zernike 相位图、真值与系数对比

In [ ]:
def to_np(t):
    if torch.is_tensor(t):
        return t.squeeze().detach().cpu().numpy()
    return np.asarray(t)

if not examples:
    print('没有示例可视化，请先运行推理单元。')
else:
    for ex in examples:
        recon = to_np(ex['recon'])
        y = to_np(ex['y']) if ex['y'] is not None else None
        zernike_indices = OPTICS_CFG.get('zernike_indices', list(range(3, 16))) if OPTICS_CFG else None
        pred_coeffs = as_zernike_coeffs(ex['pred_coeffs'], zernike_indices, OPTICS_CFG)
        true_coeffs = as_zernike_coeffs(ex['true_coeffs'], zernike_indices, OPTICS_CFG)
        H = recon.shape[-2] if recon.ndim >= 2 else recon.shape[0]
        W = recon.shape[-1] if recon.ndim >= 2 else recon.shape[1] if recon.ndim >= 2 else recon.shape[0]
        image_size = (H, W)
        pred_phase, _ = coeffs_to_wavefront(
            pred_coeffs,
            image_size,
            zernike_indices=zernike_indices,
            optics_cfg=OPTICS_CFG,
        )
        true_phase, _ = coeffs_to_wavefront(
            true_coeffs,
            image_size,
            zernike_indices=zernike_indices,
            optics_cfg=OPTICS_CFG,
        )

        # 计算系数 RMS（若有真值）
        coeffs_rms_val = coeffs_rms(pred_coeffs, true_coeffs)

        fig, axes = plt.subplots(1, 5, figsize=(15, 4), constrained_layout=True)
        axes[0].imshow(recon.squeeze(), cmap='gray')
        axes[0].set_title('network output (obj)')
        axes[0].axis('off')

        axes[1].imshow(pred_phase, cmap='RdBu')
        axes[1].set_title('predicted Zernike phase')
        axes[1].axis('off')

        if y is not None:
            axes[2].imshow(y.squeeze(), cmap='gray')
            axes[2].set_title('true obj')
        else:
            axes[2].text(0.5, 0.5, 'no true obj', ha='center')
        axes[2].axis('off')

        axes[3].imshow(true_phase, cmap='RdBu')
        axes[3].set_title('true Zernike phase')
        axes[3].axis('off')

        # 列表/表格显示系数和值
        txt = 'pred coeffs:\n' + (np.array2string(pred_coeffs, precision=3, separator=", ") if pred_coeffs is not None else 'None') + '\n\n' + 'true coeffs:\n' + (np.array2string(true_coeffs, precision=3, separator=", ") if true_coeffs is not None else 'None') + f'\n\nRMS coeffs: {coeffs_rms_val:.4f}'
        axes[4].text(0.01, 0.5, txt, va='center', fontsize=9, family='monospace')
        axes[4].axis('off')
        plt.show()

---
说明：
- 本笔记本提供一个可运行的模版，但 Zernike 基函数计算、数据字段名、模型输出格式可能需要根据你的工程/数据进行适配。
- 若需要，我可以把 Zernike 基函数改为具体实现或帮你接入你仓库中的 zernike 工具。